# Feature Engineering
This notebook serves to identify which features to split the dataset on for model training and testing.

In [0]:
%pip install pandas
%pip install numpy
%pip install scikit-learn

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [0]:
# load the preprocessed parquet file from notebook 02:
input_path = '/Volumes/workspace/default/microbiome_project_files/processed/normalized_data.parquet'
df = pd.read_parquet(input_path)

print(f"loaded {df.shape[0]} samples, {df.shape[1]} columns")

loaded 432 samples, 2219 columns


In [0]:
# keep only necessary columns
metadata_to_keep = ['sample_name', 'empo_3']
taxonomy_cols = [col for col in df.columns if col not in df.columns[:347]]  # first 347 are metadata

df_clean = df[metadata_to_keep + taxonomy_cols]

print(f"kept {df_clean.shape[1]} columns: {len(metadata_to_keep)} metadata + {len(taxonomy_cols)} taxonomy features")

kept 1874 columns: 2 metadata + 1872 taxonomy features


In [0]:
# separate features (X) and target (y)
X = df_clean.drop(['sample_name', 'empo_3'], axis=1)
y = df_clean['empo_3']
sample_names = df_clean['sample_name']

print(f"X shape: {X.shape}")
print(f"y distribution:\n{y.value_counts()}")

X shape: (432, 1872)
y distribution:
empo_3
Animal distal gut          71
Sediment (saline)          65
Plant surface              49
Soil (non-saline)          37
Water (saline)             36
Sediment (non-saline)      36
Animal corpus              33
Animal proximal gut        25
Subsurface (non-saline)    23
Water (non-saline)         22
Animal secretion           20
Fungus corpus              12
Surface (saline)            3
Name: count, dtype: int64


Since we have class imbalance (72 gut samples vs 4 surface samples), we'll want to use stratification over a random split to ensure proportional representation.

In [0]:
# train/test split with stratification
X_train, X_test, y_train, y_test, names_train, names_test = train_test_split(X, y, sample_names,
                                                                             test_size=0.2,
                                                                             random_state=42, 
                                                                             stratify=y)

print(f"train set: {X_train.shape[0]} samples")
print(f"test set: {X_test.shape[0]} samples")
print(f"\ntrain distribution:\n{y_train.value_counts()}")
print(f"\ntest distribution:\n{y_test.value_counts()}") # should be empty?

train set: 345 samples
test set: 87 samples

train distribution:
empo_3
Animal distal gut          57
Sediment (saline)          52
Plant surface              39
Soil (non-saline)          29
Sediment (non-saline)      29
Water (saline)             29
Animal corpus              26
Animal proximal gut        20
Subsurface (non-saline)    18
Water (non-saline)         18
Animal secretion           16
Fungus corpus              10
Surface (saline)            2
Name: count, dtype: int64

test distribution:
empo_3
Animal distal gut          14
Sediment (saline)          13
Plant surface              10
Soil (non-saline)           8
Sediment (non-saline)       7
Water (saline)              7
Animal corpus               7
Subsurface (non-saline)     5
Animal proximal gut         5
Water (non-saline)          4
Animal secretion            4
Fungus corpus               2
Surface (saline)            1
Name: count, dtype: int64


In [0]:
# save train and test sets for model runs
train_df = pd.concat([names_train.reset_index(drop=True), 
                      y_train.reset_index(drop=True),
                      X_train.reset_index(drop=True)], axis=1)
test_df = pd.concat([names_test.reset_index(drop=True), 
                     y_test.reset_index(drop=True),
                     X_test.reset_index(drop=True)], axis=1)

train_path = '/Volumes/workspace/default/microbiome_project_files/processed/train_data.parquet'
test_path = '/Volumes/workspace/default/microbiome_project_files/processed/test_data.parquet'

train_df.to_parquet(train_path, index=False)
test_df.to_parquet(test_path, index=False)

print(f"saved train to {train_path}")
print(f"saved test to {test_path}")

saved train to /Volumes/workspace/default/microbiome_project_files/processed/train_data.parquet
saved test to /Volumes/workspace/default/microbiome_project_files/processed/test_data.parquet
